# 🏞️ Large-Scale Scene Classification with EfficientNet-B2

**Objective:** Classify images into **397 scene categories** using transfer learning with EfficientNet-B2.

This notebook implements a complete deep learning pipeline for fine-grained scene recognition on the SUN 397 dataset, covering data preprocessing, model training with mixed-precision, and inference with test-time augmentation.

---
## 1. Imports & Configuration

Load all required libraries and define global constants. Key dependencies:
- **PyTorch** — Deep learning framework
- **timm** — Pretrained model zoo (EfficientNet-B2)
- **scikit-learn** — Stratified splitting and evaluation metrics
- **Pillow** — Image I/O and preprocessing

Reproducibility is ensured by seeding all random number generators.

In [ ]:
import os, random, numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ── Paths ─────────────────────────────────────────────────────────────
RAW_IMAGE_DIR = "/kaggle/input/sun397-scene-dataset/images"
TRAIN_CSV     = "/kaggle/input/sun397-scene-dataset/TRAIN.csv"
TEST_CSV      = "/kaggle/input/sun397-scene-dataset/TEST.csv"
RESIZED_DIR   = "/kaggle/working/images_resized"
MODEL_PATH    = "/kaggle/working/model.pth"
SUB_PATH      = "/kaggle/working/predictions.csv"

# ── Hyperparameters ───────────────────────────────────────────────────
IMG_SIZE     = 224
BATCH_SIZE   = 64
NUM_WORKERS  = 2
EPOCHS       = 30
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
print(f"GPUs    : {torch.cuda.device_count()}")

---
## 2. Image Preprocessing — Resize to Target Resolution

All raw images are resized to **224×224** pixels and saved as JPEG (quality=90) in a working directory. This one-time preprocessing step eliminates repeated I/O and decoding overhead during training.

The cell is idempotent — if images are already resized, it skips the operation.

In [ ]:
os.makedirs(RESIZED_DIR, exist_ok=True)
files = os.listdir(RAW_IMAGE_DIR)
already_done = len(os.listdir(RESIZED_DIR))

if already_done >= len(files):
    print(f"Already resized {already_done} images. Skipping.")
else:
    print(f"Resizing {len(files)} images to {IMG_SIZE}x{IMG_SIZE}...")
    for fname in tqdm(files):
        dst = os.path.join(RESIZED_DIR, fname)
        if os.path.exists(dst):
            continue
        try:
            img = Image.open(os.path.join(RAW_IMAGE_DIR, fname)).convert("RGB")
            img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            img.save(dst, format="JPEG", quality=90)
        except Exception:
            pass
    print("Resize complete.")

IMAGE_DIR = RESIZED_DIR
print(f"Using IMAGE_DIR: {IMAGE_DIR}")

---
## 3. Data Loading & Label Encoding

Load the training and test CSV files. A deterministic label-to-index mapping is constructed from the **sorted unique labels**, ensuring consistent encoding across runs.

- **TRAIN.csv** contains `IMAGE` (filename) and `LABEL` (scene category)
- **TEST.csv** contains only `IMAGE` (filenames for inference)

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
train_df.columns = [c.upper() for c in train_df.columns]
test_df.columns  = [c.upper() for c in test_df.columns]

unique_labels = sorted(train_df["LABEL"].unique())
label2idx     = {lbl: idx for idx, lbl in enumerate(unique_labels)}
idx2label     = {idx: lbl for lbl, idx in label2idx.items()}
train_df["LABEL_IDX"] = train_df["LABEL"].map(label2idx)
NUM_CLASSES = len(unique_labels)

print(f"Train: {len(train_df):,} | Test: {len(test_df):,} | Classes: {NUM_CLASSES}")
print(train_df.head(3))

---
## 4. Stratified Train / Validation Split

The training data is split into **90% training** and **10% validation** using stratified sampling. This preserves the original class distribution in both partitions, which is critical for reliable evaluation with 397 imbalanced classes.

In [ ]:
train_split, val_split = train_test_split(
    train_df, test_size=0.1,
    stratify=train_df["LABEL_IDX"], random_state=SEED
)
train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)
print(f"Train: {len(train_split):,} | Val: {len(val_split):,}")

---
## 5. Dataset Class & Data Augmentation

Define a custom PyTorch `Dataset` and image transforms:

**Training Augmentation:**
- `RandomHorizontalFlip` — Doubles effective dataset size
- `ColorJitter` — Random perturbations to brightness, contrast, saturation, and hue
- ImageNet normalization (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`)

**Validation / Test:**
- ImageNet normalization only (no augmentation)

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Images are already 224x224 so transforms are lightweight
train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.05),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])


class ImageDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_name = row["IMAGE"]
        img_path = os.path.join(self.img_dir, img_name)
        image    = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if self.is_test:
            return image, img_name
        return image, int(row["LABEL_IDX"])

---
## 6. DataLoaders

Create PyTorch DataLoaders for train, validation, and test splits. Key settings:
- **Batch size:** 64
- **Pin memory:** Enabled for faster GPU transfers
- **Persistent workers:** Enabled to avoid worker re-initialization overhead

In [ ]:
train_loader = DataLoader(
    ImageDataset(train_split, IMAGE_DIR, train_transform),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True
)
val_loader = DataLoader(
    ImageDataset(val_split, IMAGE_DIR, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True
)
test_loader = DataLoader(
    ImageDataset(test_df, IMAGE_DIR, val_transform, is_test=True),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

---
## 7. Model Architecture — EfficientNet-B2

Load a pretrained **EfficientNet-B2** backbone from `timm` and replace the classifier head with a linear layer mapping to 397 output classes.

**Training Configuration:**
- **Loss:** CrossEntropyLoss with label smoothing (ε = 0.1) to prevent overconfident predictions
- **Optimizer:** AdamW with weight decay (1e-4) for L2 regularization
- **Scheduler:** OneCycleLR with cosine annealing and 10% warm-up
- **Mixed Precision:** CUDA AMP via GradScaler for ~2x training speedup

**Model Stats:** ~8.3M parameters

In [ ]:
model = timm.create_model("efficientnet_b2", pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.1
)
scaler = torch.amp.GradScaler("cuda")

print(f"Model ready | {sum(p.numel() for p in model.parameters())/1e6:.1f}M params | {NUM_CLASSES} classes")

---
## 8. Training Loop

The training loop runs for **30 epochs** with the following per-epoch procedure:

1. **Forward pass** with mixed-precision autocast
2. **Backward pass** with scaled gradients (GradScaler)
3. **Learning rate update** via OneCycleLR (per-step scheduling)
4. **Validation** with F1 Score (Macro & Weighted) and Accuracy
5. **Checkpointing** — Save model weights when validation accuracy improves

In [ ]:
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):

    # ── Train ────────────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Ep {epoch}/{EPOCHS} Train"):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(images)
            loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss    += loss.item()
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total   += labels.size(0)

    train_acc = train_correct / train_total

    # ── Validate ─────────────────────────────────────────────────────
    model.eval()
    val_preds, val_labels_list = [], []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Ep {epoch}/{EPOCHS} Val"):
            images = images.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type="cuda"):
                outputs = model(images)
            val_preds.extend(outputs.argmax(1).cpu().numpy())
            val_labels_list.extend(labels.numpy())

    val_acc      = accuracy_score(val_labels_list, val_preds)
    val_f1_macro = f1_score(val_labels_list, val_preds, average="macro",    zero_division=0)
    val_f1_wtd   = f1_score(val_labels_list, val_preds, average="weighted", zero_division=0)

    print(f"Epoch {epoch}/{EPOCHS} | Train Acc: {train_acc:.4f} | "
          f"Val Acc: {val_acc:.4f} | F1 Macro: {val_f1_macro:.4f} | F1 Wtd: {val_f1_wtd:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"  ✔ Best model saved (val_acc={best_val_acc:.4f})")

print(f"\nDone. Best val accuracy: {best_val_acc:.4f}")

---
## 9. Inference with Test-Time Augmentation (TTA)

Load the best checkpoint and generate predictions on the test set. **Test-time augmentation** is applied by averaging logits from the original image and its horizontal flip, improving prediction robustness without additional model complexity.

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

all_preds, all_names = [], []

with torch.no_grad():
    for images, img_names in tqdm(test_loader, desc="Inference"):
        images = images.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda"):
            out1 = model(images)
            out2 = model(torch.flip(images, dims=[3]))
        preds = ((out1 + out2) / 2).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_names.extend(img_names)

print(f"Inference complete — {len(all_preds)} predictions")

---
## 10. Export Predictions

Map predicted indices back to original label names and save the results as a CSV file.

In [ ]:
submission = pd.DataFrame({
    "IMAGE": all_names,
    "LABEL": [idx2label[i] for i in all_preds]
})
submission.to_csv(SUB_PATH, index=False)

print(f"Saved: {SUB_PATH}")
print(submission.head(10))